In [1]:
# !pip install ranx

Looking in indexes: https://artifactory.tcsbank.ru/artifactory/api/pypi/python-all/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 8.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 41.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 50.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 26.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.0/310.0 kB 21.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.1/252.1 kB 26.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 46.9 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 36.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 42.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 8.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "litsearch"


In [3]:
from src.evaluation.build_qrels import build_qrels_from_queries
from src.evaluation.build_runs import (
    build_run_from_candidates,
    build_run_from_rerank,
)
from src.evaluation.evaluate import evaluate_runs


In [4]:
qrels = build_qrels_from_queries(DATA_DIR / "queries.jsonl")


In [5]:
import json

empty_coarse = 0
with open(DATA_DIR / "coarse_rerank.jsonl") as f:
    for line in f:
        if not json.loads(line)["ranked_doc_ids"]:
            empty_coarse += 1

empty_fine = 0
with open(DATA_DIR / "fine_rerank.jsonl") as f:
    for line in f:
        if not json.loads(line)["final_ranked_doc_ids"]:
            empty_fine += 1

empty_coarse, empty_fine


(325, 597)

In [7]:
# BLOCK 1 — BM25 only

runs = {
    "BM25": build_run_from_candidates(DATA_DIR / "candidates.jsonl")
}

results = evaluate_runs(runs, DATA_DIR / "queries.jsonl")
results

TypeError: evaluate_runs() missing 1 required positional argument: 'metrics'

In [5]:
runs = {
    "BM25": build_run_from_candidates(DATA_DIR / "candidates.jsonl"),
    "Coarse": build_run_from_rerank(
        DATA_DIR / "coarse_rerank.jsonl",
        key="ranked_doc_ids",
    ),
    "Fine": build_run_from_rerank(
        DATA_DIR / "fine_rerank.jsonl",
        key="final_ranked_doc_ids",
    ),
}


ValueError: max() arg is an empty sequence